# OpenAI QA inference

Pinned model `gpt-5.4-2026-03-05` with reasoning fully **off** (`reasoning_effort="none"`).

The 124 validation examples are processed with at most 16 concurrent API requests.

In [1]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from datasets import load_from_disk
from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

load_dotenv()

/Users/faizanfaisal/Laptop_Data/masters/masters_codebases/LEGAL-UQA/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
dataset_dict = load_from_disk("../../data/complete_dataset/")

print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['question_eng', 'question_urdu', 'context_eng', 'context_urdu', 'answer_eng', 'answer_urdu', 'context_index', '__index_level_0__'],
        num_rows: 495
    })
    validation: Dataset({
        features: ['question_eng', 'question_urdu', 'context_eng', 'context_urdu', 'answer_eng', 'answer_urdu', 'context_index', '__index_level_0__'],
        num_rows: 124
    })
})


In [3]:
validation_df = pd.DataFrame(dataset_dict["validation"])
df = validation_df[["question_urdu", "answer_urdu"]].copy()
df["id"] = range(len(df))
df.rename(
    columns={"question_urdu": "question", "answer_urdu": "answers"},
    inplace=True,
)
df = df[["id", "question", "answers"]]

display(df)

,id,answers
0,0,اسلامی کونسل کا کوئی رکن تین سال کی مدت کے لیے...
1,1,پاکستان میں یوم آغاز سے قبل قبائلی علاقے جات م...
2,2,نہیں، کسی عدالت عالیہ کے جج کا اس کی رضامندی ک...
3,3,وفاقی حکومت یا، جیسی بھی صورت ہو، صوبائی حکومت...
4,4,کسی رکن کو منحرف قرار دینے سے پہلے، پارٹی کے س...
...,...,...
119,119,وزیر اعظم مشترکہ مفادات کی کونسل کا چیئرمین ہو...
120,120,نہیں، کسی شخص کو شہادت حاصل کرنے کی غرض سے اذی...
121,121,سینیٹ میں اسپیکر اور ڈپٹی اسپیکر کے مساوی عہدے...
122,122,مالی بل کی ابتدا قومی اسمبلی میں ہوگی۔


In [4]:
MODEL_NAME = "gpt-5.4-2026-03-05"
MAX_WORKERS = 16
client = OpenAI(max_retries=5, timeout=120.0)


def get_urdu_response(context, question):
    completion = client.chat.completions.create(
        model=MODEL_NAME,
        reasoning_effort="none",  # fully disable reasoning
        max_completion_tokens=1024,
        messages=[
            {"role": "system", "content": "You are a Pakistani legal expert who speaks in Urdu. You only respond in Urdu script."},
            {"role": "user", "content": f"""Below is a context provided, followed by a question. Your task is to answer the question based on the given context. The answer should consist only of the response to the question in Urdu and should not include any other information or commentary.

    Context: ```{context}```

    Question: ```{question}```

    Answer:"""},
        ],
    )

    return completion.choices[0].message.content

In [5]:
validation_examples = list(dataset_dict["validation"])
predicted_answers = [None] * len(validation_examples)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_index = {
        executor.submit(
            get_urdu_response,
            example["context_urdu"],
            example["question_urdu"],
        ): index
        for index, example in enumerate(validation_examples)
    }

    for future in tqdm(
        as_completed(future_to_index),
        total=len(future_to_index),
        desc=f"Processing {MODEL_NAME} responses",
        unit="item",
    ):
        index = future_to_index[future]
        predicted_answers[index] = future.result()

assert all(answer is not None for answer in predicted_answers)


Processing gpt-5.4 responses:   0%|          | 0/124 [00:00<?, ?item/s]


Processing gpt-5.4 responses:   1%|          | 1/124 [00:02<05:28,  2.67s/item]


Processing gpt-5.4 responses:   2%|▏         | 2/124 [00:03<02:55,  1.44s/item]


Processing gpt-5.4 responses:   5%|▍         | 6/124 [00:03<00:42,  2.78item/s]


Processing gpt-5.4 responses:   7%|▋         | 9/124 [00:03<00:26,  4.40item/s]


Processing gpt-5.4 responses:  10%|▉         | 12/124 [00:03<00:20,  5.42item/s]

... (52 outputs left out)


Processing gpt-5.4 responses:  96%|█████████▌| 119/124 [00:16<00:00, 12.53item/s]


Processing gpt-5.4 responses:  98%|█████████▊| 121/124 [00:16<00:00, 11.61item/s]


Processing gpt-5.4 responses:  99%|█████████▉| 123/124 [00:17<00:00, 10.25item/s]


Processing gpt-5.4 responses: 100%|██████████| 124/124 [00:17<00:00,  7.03item/s]

In [6]:
predicted_answers[:5]

['اسلامی کونسل کا رکن تین سال کی مدت کے لیے اپنے عہدے پر فائز رہتا ہے۔',
 'پاکستان کے وہ علاقے جات جو یوم آغاز سے عین قبل قبائلی علاقے جات تھے، ان میں بلوچستان اور خیبر پختونخوا کے قبائلی علاقے جات، اور امب، چترال، دیر اور سوات کی سابقہ ریاستیں شامل تھیں۔',
 'نہیں، کسی عدالت عالیہ کے جج کا اس کی رضامندی کے بغیر کسی دوسری عدالت عالیہ میں تبادلہ نہیں کیا جا سکتا۔',
 'وفاقی حکومت یا، حسبِ معاملہ، صوبائی حکومت کے پاس۔',
 'پارٹی کے سربراہ کو اعلان کرنے سے پہلے متعلقہ رکن کو اظہارِ وجوہ کا موقع فراہم کرنا چاہیے کہ کیوں نہ اس کے خلاف انحراف کا اعلان کر دیا جائے۔']

In [7]:
df["requested_model"] = MODEL_NAME
df["model_version"] = MODEL_NAME
df["reasoning_mode"] = "none"
df["prediction_text_gpt_5_4"] = predicted_answers

In [8]:
display(df.head())

,id,answers,prediction_text_gpt_5_4
0,0,اسلامی کونسل کا کوئی رکن تین سال کی مدت کے لیے...,اسلامی کونسل کا رکن تین سال کی مدت کے لیے اپنے...
1,1,پاکستان میں یوم آغاز سے قبل قبائلی علاقے جات م...,پاکستان کے وہ علاقے جات جو یوم آغاز سے عین قبل...
2,2,نہیں، کسی عدالت عالیہ کے جج کا اس کی رضامندی ک...,نہیں، کسی عدالت عالیہ کے جج کا اس کی رضامندی ک...
3,3,وفاقی حکومت یا، جیسی بھی صورت ہو، صوبائی حکومت...,وفاقی حکومت یا، حسبِ معاملہ، صوبائی حکومت کے پاس۔
4,4,کسی رکن کو منحرف قرار دینے سے پہلے، پارٹی کے س...,پارٹی کے سربراہ کو اعلان کرنے سے پہلے متعلقہ ر...


In [9]:
df.to_csv("../../data/model_qa_responses/gpt-5.4-responses.csv", index=False)